In [1]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'


import jax
import jax.numpy as jnp

from jax import config
config.update("jax_enable_x64", True)

In [2]:
from LIMxCMBL.init import *
from LIMxCMBL.kernels import *
from LIMxCMBL.experiments import *

from scipy.interpolate import interp1d, LinearNDInterpolator
from scipy.integrate import quad, quad_vec, trapezoid, qmc_quad
from scipy.stats import qmc

from tqdm import trange

/home/users/delon/.local/lib/python3.9/site-packages/astropy/units/quantity.py:673: RuntimeWarning: invalid value encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


In [3]:
_tmp_ks = np.logspace(-10, 5, 100000)
_tmp_Pk = np.zeros_like(_tmp_ks)

for k_idx, k in enumerate(_tmp_ks):
    _tmp_Pk[k_idx] = ccl.linear_matter_power(cosmo, k, 1)
    


_ells = jnp.logspace(1, np.log10(5000), 100)
_ells = _ells.reshape(-1,1)

In [4]:
kernels = {}
kernels['CII'] = np.array(KI)
kernels['CO'] = np.array(KI_CO)
kernels['Lya'] = np.array(KI_Lya)
kernels['HI'] = np.array(KI_HI)

In [5]:
import matplotlib.pyplot as plt

In [6]:
inner_dkparp_integral = np.load('/oak/stanford/orgs/kipac/users/delon/LIMxCMBL/inner_dkparp_integral.npy')
inner_dkparp_integral = inner_dkparp_integral.astype(np.float64)
inner_dkparp_integral = np.moveaxis(inner_dkparp_integral, 0, -1)

In [13]:
inner_dkparp_integral.shape

(256, 128, 100)

In [26]:
from interpax import interp1d as interp1dx
from interpax import interp2d

In [106]:
for experiment in experiments:
    n_bins = 100
    
    if(experiment == 'SPHEREx'):
        n_bins = 15
        
        
#     for Lambda_idx in np.hstack([-1, experiments[experiment]['Lambda_idxs']]):        
    for Lambda_idx in np.hstack(experiments[experiment]['Lambda_idxs']):      

        Lambda = Lambdas[Lambda_idx]
        Lambda = 0.0
        if(Lambda_idx >= 0):
            Lambda = Lambdas[Lambda_idx]

        oup_fname = '/scratch/users/delon/LIMxCMBL/IHiKappa/'
        oup_fname += '%s_LIMBER_IHik_idx_%d.npy'%(experiment, Lambda_idx, )

        zmin = experiments[experiment]['zmin']
        zmax = experiments[experiment]['zmax']

        line_str = experiments[experiment]['line_str']


        chimin = ccl.comoving_angular_distance(cosmo, 1/(1+zmin))
        chimax = ccl.comoving_angular_distance(cosmo, 1/(1+zmax))


        chi_bin_edges = np.linspace(chimin*(1+1e-8), chimax*(1 - 1e-8), n_bins + 1)
        chi_bin_centers = (chi_bin_edges[1:] + chi_bin_edges[:-1])/2
        dchi_binned = np.mean(np.diff(chi_bin_edges))


        m_for_LO = 0
        while(m_for_LO * 2 * jnp.pi / (chimax - chimin) < Lambda):
            m_for_LO += 1
        m_for_LO -= 1
        print('m for low pass is', m_for_LO)


        @jax.jit
        def jax_lo_DFT(alpha, m):
            _L = chimax - chimin
            return jnp.where(
                (jnp.abs((jnp.abs(alpha)-_L) / _L) < 1e-5) | (jnp.abs(alpha / _L) < 1e-5),
                (1 + 2 * m) / _L,
                jnp.sin(jnp.pi/_L * alpha * (1 + 2 * m)) / jnp.sin(jnp.pi/_L * alpha) * 1/_L
            )

        @jax.jit
        def f_KILo(chip, external_chi, m):
            return (jnp.interp(x = chip, xp = chis, 
                                 fp = _KI, left = 0, right = 0) 
                    * jax_lo_DFT(alpha = external_chi - chip, 
                                           m = m))



        _KI = kernels[line_str]

        
        @jax.jit
        def f_unfiltered_integrand(chi, _chib):
            _curr_KI = 2 * jnp.interp(x = chi, xp = chis, fp = _KI, left = 0, right = 0)
            _delta = jnp.abs(1 - chi / _chib)
            _delta = jnp.where(_delta < 1e-6, 1e-6,
                               jnp.where(_delta > 0.7, 
                                     0.7,
                                     _delta))
            unfiltered_integrand = (_curr_KI.T
                                    * jnp.interp(x = 2*_chib - chi, 
                                                xp = chis, fp = Wk * Dz, 
                                                left = 0, right = 0).T
                                    * interp2d(xq = _chib.reshape(-1), yq=jnp.log(_delta).reshape(-1), 
                                               x = chibs, y = jnp.log(deltas), f=inner_dkparp_integral,
                                               method='linear',)
                                    / _chib.T**2)
            return unfiltered_integrand
 

        @jax.jit
        def _f_filtered_integrand(chi, _chib):
            plus = _chib*(1+deltas.reshape(-1, 1))
            mins = _chib*(1-deltas.reshape(-1, 1))
            _idxs_mins = (chimin <= mins) & (mins <= chimax)
            _idxs_plus = (chimin <= plus) & (plus <= chimax)

            _interm  = jnp.where(_idxs_plus,
                                 f_KILo(plus, 
                                        external_chi = chi,
                                        m=m_for_LO) 
                                 * jnp.interp(x = mins,
                                              xp = chis, fp = Wk * Dz, 
                                              left = 0, right = 0),
                                 0)
    
            _interm += jnp.where(_idxs_mins,
                                 f_KILo(mins, 
                                        external_chi = chi,
                                        m=m_for_LO) 
                                 * jnp.interp(x = plus,
                                              xp = chis, fp = Wk * Dz, 
                                              left = 0, right = 0),
                                 0)

            _factor = (2 / _chib)
            _factor = _factor * deltas.reshape(-1, 1)
            _factor = jnp.einsum('dp, pdl->pld', _factor, 
                                 interp1dx(xq = _chib.reshape(-1),
                                           x = chibs, 
                                           f = inner_dkparp_integral,
                                           method='linear',)
                                )
            
            _interm  = jnp.einsum('dp,pld->pld', _interm, _factor)
            LO_integrand = jnp.trapezoid(x = np.log(deltas), y = _interm, axis=-1)
            return LO_integrand

        @jax.jit
        def f_filtered_integrand(x):
            chi, _chib = x[0], x[1]

            chi = chi.reshape(1, -1)
            _chib = _chib.reshape(1, -1)
            return  f_unfiltered_integrand(chi=chi, _chib=_chib) - _f_filtered_integrand(chi=chi, _chib=_chib)
        
        #do integral
        IHi_kappa = np.zeros((len(ells), n_bins))
        for chi_idx in trange(n_bins):
            qrng = qmc.Halton(d = 2)

            l, r = chi_bin_edges[chi_idx], chi_bin_edges[chi_idx+1]

            def _rng_spawn(rng, n_children):
                bg = rng._bit_generator
                ss = bg._seed_seq
                child_rngs = [np.random.Generator(type(bg)(child_ss))
                              for child_ss in ss.spawn(n_children)]
                return child_rngs

            n_estimates = 2**3
            n_points = 2**13
            estimates = np.zeros((n_estimates, 100))

            rngs = _rng_spawn(qrng.rng, n_estimates)

            for i in range(n_estimates):
                sample = qrng.random(n = n_points)
                sample_bin = sample[:, -1]

                _chis = qmc.scale(jnp.array([sample_bin]), l, r)
#                 estimates[i] = jnp.mean(f_unfiltered_integrand(_chis), axis = -1)

                if(Lambda_idx > 0):
                    a = np.array([l, chimin,])
                    b = np.array([r, chimax,])

                    #only worry about measure for dchib integral
                    #since we want averages in chi bins
                    dA = (chimax - chimin) / n_points

                    x = jnp.array(qmc.scale(sample, a, b)).T
                    estimates[i] = jnp.sum(f_filtered_integrand(x).T * dA, axis = -1)


                qrng = type(qrng)(seed=rngs[i], **qrng._init_quad)
            integral = jnp.mean(estimates, axis=0)
            standard_error = jnp.std(estimates, axis = 0, ddof = 1)
            IHi_kappa[:, chi_idx] = integral
            if(chi_idx == n_bins//2):
                print(standard_error/integral)
            
#         jnp.save(oup_fname, IHi_kappa)
        plt.imshow(IHi_kappa)
        plt.title('%s, %.2f'%(experiment, Lambda))
        plt.show()
#         print(oup_fname)

m for low pass is 0


  0%|          | 0/100 [00:00<?, ?it/s]

(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)


  1%|          | 1/100 [00:27<44:59, 27.27s/it]

(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)
(8192, 100)


  1%|          | 1/100 [00:49<1:22:23, 49.94s/it]


KeyboardInterrupt: 